<!--nav--> [🗺 Learning path](README.md) · **26/40** · ◀ [Speculative Decoding](./Speculative_Decoding_Advanced_Serving.ipynb) · [Reading the Logs](./Serving_Logs_Observability.ipynb) ▶

# Serving Internals, Visualized — Interactive D3 Explanations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugeerth/gpu-training-notebooks/blob/main/Serving_Internals_Visualized_D3.ipynb)

Notebooks 21–24 *measured* the serving stack. This one lets you **watch it move**. Every concept
gets an interactive [D3.js](https://d3js.org) visualization you can scrub, animate, and poke at —
because a scheduler is easier to understand as an animation than as a paragraph.

**No GPU needed.** Every simulation is pure Python; the browser does the drawing. Run it on the
free Colab CPU runtime, your laptop, anywhere.

| Viz | What you'll *see* | The idea from |
|---|---|---|
| **1** | KV-cache memory explorer — drag the context slider, watch GPUs fill up | notebook 21 |
| **2** | Static vs continuous batching — an animated two-lane Gantt with live utilization | notebook 21 |
| **3** | PagedAttention block pool vs a contiguous allocator, frame by frame | notebook 22 |
| **4** | Speculative decoding accept/reject strip with α and k sliders | notebook 24 |

### How every cell works (the pattern)

```
Python cell:  simulate / compute  →  plain dicts & lists  →  json
                                                              ↓
D3 cell:      show_d3(js_code, data)  →  HTML+SVG in the output, data injected as JSON
```

Python owns the *truth* (the simulation), JavaScript owns the *pixels*. That split is worth
copying: you can unit-test the simulation, and the viz can't lie about the data.

In [ ]:
# The one helper this notebook is built on: render a D3 visualization from Python data.
# D3 v7 loads from a CDN once per output cell (Colab sandboxes each output in its own iframe).
import json, uuid
from IPython.display import HTML, display

D3_URL = "https://cdn.jsdelivr.net/npm/d3@7/dist/d3.min.js"

def show_d3(js, data=None, height=420):
    # Runs `js` with four variables already in scope:
    #   d3    - the D3 v7 library
    #   root  - a d3 selection of this cell's own container <div>
    #   data  - your Python object, JSON-round-tripped
    #   W, H  - the container's pixel width and the height you asked for
    div = f"viz_{uuid.uuid4().hex[:10]}"
    html = f'''
<div id="{div}" style="width:100%;max-width:920px;font-family:system-ui,sans-serif"></div>
<script>
(function() {{
  function run() {{
    const d3 = window.d3;
    const root = d3.select("#{div}");
    const data = {json.dumps(data)};
    const W = (document.getElementById("{div}").clientWidth || 880), H = {height};
    try {{ {js} }} catch (e) {{ root.append("pre").style("color","crimson").text("viz error: " + e); }}
  }}
  if (window.d3) run();
  else {{
    const s = document.createElement("script");
    s.src = "{D3_URL}";
    s.onload = run;
    s.onerror = () => document.getElementById("{div}").textContent =
        "Could not load D3 from the CDN - check your network and re-run this cell.";
    document.head.appendChild(s);
  }}
}})();
</script>'''
    display(HTML(html))

print("show_d3 ready - every visualization below is: python data -> show_d3(js, data)")

## Viz 1 · The KV-cache memory explorer

Step 1 of understanding serving is *feeling* how fast the KV cache eats a GPU. The formula (from
[notebook 21](./Serving_Fundamentals_KV_Cache_Batching.ipynb)):

```
KV bytes/token = 2 × n_layers × n_kv_heads × head_dim × 2 bytes     (fp16, K and V)
```

First compute the truth in Python — same architectures as notebook 21's calculator:

In [ ]:
# Step 1: the data. Real architecture numbers -> KV KB/token (MLA stores a compressed latent).
ARCH = {
    "Qwen2.5-0.5B (GQA)":  dict(layers=24, kv_heads=2,  head_dim=64,  mla=None),
    "Llama-3.1-8B (GQA)":  dict(layers=32, kv_heads=8,  head_dim=128, mla=None),
    "Qwen2.5-72B (GQA)":   dict(layers=80, kv_heads=8,  head_dim=128, mla=None),
    "GPT-3-175B (MHA)":    dict(layers=96, kv_heads=96, head_dim=128, mla=None),
    "DeepSeek-V3 (MLA)":   dict(layers=61, kv_heads=None, head_dim=None, mla=512 + 64),
}

kv_data = []
for name, a in ARCH.items():
    per_tok = (a["layers"] * a["mla"] * 2) if a["mla"] else \
              (2 * a["layers"] * a["kv_heads"] * a["head_dim"] * 2)
    kv_data.append({"name": name, "kb_per_token": per_tok / 1024})
    print(f"{name:<22} {per_tok/1024:8.1f} KB/token")

In [ ]:
# Step 2: the pixels. Drag the sliders; bars show KV GB per conversation,
# labels show how many such conversations fit in the chosen GPU's free memory.
JS = r'''
const M = {top: 30, right: 130, bottom: 46, left: 190};
const iw = W - M.left - M.right, ih = H - M.top - M.bottom;

const controls = root.append("div").style("margin", "4px 0 10px 0");
function slider(label, min, max, value, step) {
  const wrap = controls.append("label").style("margin-right", "24px").style("font-size", "13px");
  wrap.append("span").text(label + " ");
  const out = wrap.append("b");
  const inp = wrap.append("input").attr("type", "range")
      .attr("min", min).attr("max", max).attr("step", step).attr("value", value)
      .style("vertical-align", "middle").style("margin-left", "6px");
  return {inp, out};
}
const ctx = slider("context tokens:", 9, 17, 13, 1);      // 2^9=512 .. 2^17=131072
const gpu = slider("GPU free VRAM (GB):", 8, 80, 16, 8);

const svg = root.append("svg").attr("width", W).attr("height", H)
    .append("g").attr("transform", `translate(${M.left},${M.top})`);
const y = d3.scaleBand().domain(data.map(d => d.name)).range([0, ih]).padding(0.25);
const x = d3.scaleLog().range([0, iw]);
const xAxisG = svg.append("g").attr("transform", `translate(0,${ih})`);
svg.append("g").call(d3.axisLeft(y).tickSize(0)).select(".domain").remove();
svg.append("text").attr("x", iw / 2).attr("y", ih + 38).attr("text-anchor", "middle")
    .style("font-size", "12px").style("fill", "#888").text("KV cache per conversation (GB, log scale)");

const bars = svg.selectAll(".bar").data(data).join("rect")
    .attr("y", d => y(d.name)).attr("height", y.bandwidth()).attr("x", 0)
    .attr("fill", (d, i) => d3.schemeTableau10[i]);
const labels = svg.selectAll(".lab").data(data).join("text")
    .attr("y", d => y(d.name) + y.bandwidth() / 2 + 4).style("font-size", "12px");

function update() {
  const ctxTokens = 2 ** +ctx.inp.node().value;
  const gpuGB = +gpu.inp.node().value;
  ctx.out.text(ctxTokens.toLocaleString());
  gpu.out.text(gpuGB);
  const rows = data.map(d => ({...d, gb: d.kb_per_token * ctxTokens / 1e6}));
  x.domain([Math.max(1e-3, d3.min(rows, d => d.gb) * 0.5), d3.max(rows, d => d.gb) * 2.5]);
  xAxisG.call(d3.axisBottom(x).ticks(6, ".2~s"));
  bars.data(rows).transition().duration(250).attr("width", d => Math.max(1, x(d.gb)));
  labels.data(rows).transition().duration(250)
      .attr("x", d => Math.max(1, x(d.gb)) + 8)
      .text(d => {
        const fits = Math.floor(gpuGB / d.gb);
        return `${d.gb >= 10 ? d.gb.toFixed(0) : d.gb.toPrecision(2)} GB -> fits ${fits >= 1 ? fits : "0 (!)"}`;
      })
      .style("fill", d => Math.floor(gpuGB / d.gb) === 0 ? "crimson" : "#555");
}
ctx.inp.on("input", update); gpu.inp.on("input", update);
update();
'''
show_d3(JS, kv_data, height=330)

**What to notice.** Slide context to 128k: the MHA-era model needs *hundreds of GB for one
conversation* — architecturally impossible to serve long-context. Slide down to 512: suddenly
everything fits hundreds of times. And DeepSeek's MLA beats even aggressive GQA. When a provider
prices long-context 10× higher, this chart is why.

## Viz 2 · Continuous batching, animated

Notebook 21's simulator printed *summary numbers* (98% vs 39% slot utilization). Here is the same
simulation as a living timeline. Each colored bar is a request occupying a GPU slot; the sweep line
is "now". **Top lane: static batching** (the whole batch waits for its slowest member — watch the
white gaps). **Bottom: continuous batching** (finished slot → next request, instantly).

In [ ]:
# Step 1: the simulation - identical rules to notebook 21, but now we RECORD every placement.
import random
random.seed(0)

N_REQ, SLOTS = 60, 8
lengths = [min(180, max(4, int(random.lognormvariate(3.3, 0.8)))) for _ in range(N_REQ)]

def simulate(policy):
    queue = list(range(N_REQ)); active = {}; t = 0
    placements, util = [], []          # placements: one bar per (request, slot, start, end)
    free_slots = list(range(SLOTS))
    while queue or active:
        if policy == "static" and not active:
            for s in list(free_slots):
                if not queue: break
                r = queue.pop(0); active[r] = [lengths[r], s]; free_slots.remove(s)
                placements.append({"req": r, "slot": s, "start": t, "len": lengths[r]})
        elif policy == "continuous":
            while free_slots and queue:
                r = queue.pop(0); s = free_slots.pop(0); active[r] = [lengths[r], s]
                placements.append({"req": r, "slot": s, "start": t, "len": lengths[r]})
        t += 1
        util.append(len(active) / SLOTS)
        for r in list(active):
            active[r][0] -= 1
            if active[r][0] == 0:
                free_slots.append(active[r][1]); free_slots.sort(); del active[r]
    return {"placements": placements, "util": util, "makespan": t}

sim = {"static": simulate("static"), "continuous": simulate("continuous"),
       "slots": SLOTS, "n_req": N_REQ}
print(f"static:     {sim['static']['makespan']} steps, "
      f"avg util {sum(sim['static']['util'])/len(sim['static']['util']):.0%}")
print(f"continuous: {sim['continuous']['makespan']} steps, "
      f"avg util {sum(sim['continuous']['util'])/len(sim['continuous']['util']):.0%}")

In [ ]:
# Step 2: the animation. ▶ plays a sweep line over both schedules; counters update live.
JS = r'''
const M = {top: 26, right: 16, bottom: 26, left: 84};
const laneH = 118, gap = 66;
const iw = W - M.left - M.right;
const T = Math.max(data.static.makespan, data.continuous.makespan);
const x = d3.scaleLinear().domain([0, T]).range([0, iw]);
const color = i => d3.schemeTableau10[i % 10];

const bar = root.append("div").style("margin-bottom", "6px");
const btn = bar.append("button").text("▶ play").style("font-size", "14px");
const stat = bar.append("span").style("margin-left", "16px").style("font-size", "13px");

const svg = root.append("svg").attr("width", W).attr("height", 2 * laneH + gap + M.top + M.bottom);
const panels = [["static", data.static, M.top], ["continuous", data.continuous, M.top + laneH + gap]];
const counters = {};
for (const [name, d, top] of panels) {
  const g = svg.append("g").attr("transform", `translate(${M.left},${top})`);
  const y = d3.scaleBand().domain(d3.range(data.slots)).range([0, laneH]).padding(0.18);
  g.append("text").attr("x", -8).attr("y", -8).attr("text-anchor", "start")
      .style("font-weight", 700).style("font-size", "13px")
      .text(name.toUpperCase() + " batching");
  counters[name] = g.append("text").attr("x", iw).attr("y", -8).attr("text-anchor", "end")
      .style("font-size", "12px").style("fill", "#666");
  g.append("g").attr("transform", `translate(0,${laneH})`)
      .call(d3.axisBottom(x).ticks(8)).style("font-size", "10px");
  g.selectAll("lane").data(d3.range(data.slots)).join("rect")
      .attr("x", 0).attr("y", s => y(s)).attr("width", iw).attr("height", y.bandwidth())
      .attr("fill", "#f2f2f2");
  g.selectAll("bar").data(d.placements).join("rect")
      .attr("x", p => x(p.start)).attr("y", p => y(p.slot))
      .attr("width", p => Math.max(1, x(p.start + p.len) - x(p.start)))
      .attr("height", y.bandwidth())
      .attr("fill", p => color(p.req)).attr("opacity", 0.9)
      .append("title").text(p => `req ${p.req}: ${p.len} tokens (slot ${p.slot}, t=${p.start})`);
  g.append("line").attr("class", "cursor").attr("y1", 0).attr("y2", laneH)
      .attr("stroke", "black").attr("stroke-width", 1.5);
}

let t = 0, timer = null;
function frame() {
  t = (t + 1) % (T + 30);
  const tc = Math.min(t, T);
  svg.selectAll(".cursor").attr("x1", M.left + x(tc)).attr("x2", M.left + x(tc))
     .attr("transform", null).attr("x1", x(tc)).attr("x2", x(tc));
  for (const [name, d] of [["static", data.static], ["continuous", data.continuous]]) {
    const u = d.util[Math.min(tc, d.util.length - 1)];
    const done = d.placements.filter(p => p.start + p.len <= tc).length;
    counters[name].text(`t=${tc}  ·  busy slots: ${(u * 100).toFixed(0)}%  ·  finished: ${done}/${data.n_req}` +
                        (tc >= d.makespan ? "  ·  DONE" : ""));
  }
  stat.text(`grey = an idle slot that static batching cannot refill until the whole batch drains`);
}
btn.on("click", () => {
  if (timer) { timer.stop(); timer = null; btn.text("▶ play"); }
  else { timer = d3.interval(frame, 40); btn.text("⏸ pause"); }
});
frame();
'''
show_d3(JS, sim, height=400)

**What to notice, step by step.**
1. Press play and watch the **top lane** around each batch boundary: one long bar keeps running
   while every other slot sits grey. That grey area *is* the throughput you lose.
2. The **bottom lane** has almost no grey: the moment a bar ends, a new color begins. Same
   requests, same slot count — the finished counter just runs ahead.
3. Hover any bar: requests differ wildly in length (lognormal, like real traffic). The *variance*
   is what static batching cannot handle — with uniform lengths the two lanes would look identical.

## Viz 3 · PagedAttention vs the contiguous allocator

The vLLM notebook *told* you contiguous KV allocation wastes 60–80% of memory. Now watch both
allocators run on identical traffic. Every square is one 16-token KV block:

- **red-striped** = *reserved but unused* — a contiguous allocator must reserve `max_len` up front
- **colored** = actually holding a request's KV
- **grey** = free

In [ ]:
# Step 1: simulate both allocators over identical request traffic, snapshot the pool each step.
import random
random.seed(1)

POOL, BLK, MAXLEN = 180, 16, 128           # 180 blocks of 16 tokens; requests may grow to 128
RES_BLKS = MAXLEN // BLK                   # contiguous allocator reserves 8 blocks per request

def make_traffic(n=120):
    reqs, t = [], 0
    for i in range(n):
        t += random.choice([0, 0, 1])                       # bursty arrivals
        reqs.append({"id": i, "arrive": t, "len": random.randint(8, MAXLEN)})
    return reqs

def run(policy, reqs, steps=170):
    pool = [None] * POOL                   # each entry: (req_id, used_flag) or None
    active, queue, done = {}, [], 0
    frames, waste = [], []
    ri = 0
    for t in range(steps):
        while ri < len(reqs) and reqs[ri]["arrive"] <= t:
            queue.append(dict(reqs[ri])); ri += 1
        for q in list(queue):              # try to admit
            if policy == "contig":
                run_start = next((i for i in range(POOL - RES_BLKS + 1)
                                  if all(pool[j] is None for j in range(i, i + RES_BLKS))), None)
                if run_start is None: break
                blocks = list(range(run_start, run_start + RES_BLKS))
            else:
                free = [i for i, b in enumerate(pool) if b is None]
                if not free: break
                blocks = [free[0]]         # paged: start with ONE block, grow on demand
            for b in blocks: pool[b] = [q["id"], False]
            active[q["id"]] = {"left": q["len"], "tok": 0, "blocks": blocks}
            queue.remove(q)
        for rid in list(active):           # one decode step: everyone gains a token
            a = active[rid]; a["tok"] += 1; a["left"] -= 1
            need = -(-a["tok"] // BLK)     # ceil
            if policy == "paged" and need > len(a["blocks"]):
                free = [i for i, b in enumerate(pool) if b is None]
                if free:
                    a["blocks"].append(free[0]); pool[free[0]] = [rid, False]
            for k, b in enumerate(a["blocks"]):
                pool[b][1] = (k < need)    # a block is "used" once tokens have reached it
            if a["left"] <= 0:
                for b in a["blocks"]: pool[b] = None
                del active[rid]; done += 1
        frames.append([0 if b is None else (2 * (b[0] % 10) + (3 if b[1] else 2)) for b in pool])
        reserved = sum(1 for b in pool if b is not None)
        used = sum(1 for b in pool if b is not None and b[1])
        waste.append({"reserved": reserved, "used": used, "active": len(active),
                      "queued": len(queue), "done": done})
    return {"frames": frames, "waste": waste}

traffic = make_traffic()
alloc = {"contig": run("contig", traffic), "paged": run("paged", traffic),
         "pool": POOL, "cols": 20}
w_c = alloc["contig"]["waste"][-1]; w_p = alloc["paged"]["waste"][-1]
print(f"after 170 steps - contiguous: {w_c['done']} requests done | paged: {w_p['done']} done")
mid_c = alloc["contig"]["waste"][85]; mid_p = alloc["paged"]["waste"][85]
print(f"mid-run waste  - contiguous: {mid_c['reserved']-mid_c['used']} blocks reserved-but-unused"
      f" | paged: {mid_p['reserved']-mid_p['used']}")

In [ ]:
# Step 2: side-by-side pool animation with a scrubber.
JS = r'''
const cols = data.cols, rows = Math.ceil(data.pool / cols), cell = Math.min(16, (W/2 - 90) / cols);
const T = data.contig.frames.length;

const bar = root.append("div").style("margin-bottom", "6px");
const btn = bar.append("button").text("▶ play");
const scrub = bar.append("input").attr("type", "range").attr("min", 0).attr("max", T - 1)
    .attr("value", 0).style("width", "300px").style("vertical-align", "middle")
    .style("margin-left", "12px");
const tlab = bar.append("b").style("margin-left", "8px");

const svg = root.append("svg").attr("width", W).attr("height", rows * cell + 92);
const defs = svg.append("defs");
defs.append("pattern").attr("id", "stripe").attr("width", 5).attr("height", 5)
    .attr("patternUnits", "userSpaceOnUse").attr("patternTransform", "rotate(45)")
  .append("rect").attr("width", 2.5).attr("height", 5).attr("fill", "#e57373");

const panels = {};
[["contig", "CONTIGUOUS (reserve max_len up front)", 0],
 ["paged", "PAGED (grow one block at a time)", W / 2]].forEach(([key, title, xoff]) => {
  const g = svg.append("g").attr("transform", `translate(${xoff + 45},34)`);
  g.append("text").attr("y", -18).style("font-weight", 700).style("font-size", "12.5px").text(title);
  const cells = g.selectAll("c").data(d3.range(data.pool)).join("rect")
      .attr("x", i => (i % cols) * cell).attr("y", i => Math.floor(i / cols) * cell)
      .attr("width", cell - 1.5).attr("height", cell - 1.5).attr("rx", 2);
  const label = g.append("text").attr("y", rows * cell + 20).style("font-size", "12px");
  const label2 = g.append("text").attr("y", rows * cell + 38).style("font-size", "12px").style("fill", "#666");
  panels[key] = {cells, label, label2};
});

function render(t) {
  tlab.text("step " + t);
  scrub.property("value", t);
  for (const key of ["contig", "paged"]) {
    const f = data[key].frames[t], w = data[key].waste[t];
    panels[key].cells
        .attr("fill", v => f[v] === 0 ? "#e8e8e8"
              : (f[v] % 2 === 0 ? "url(#stripe)" : d3.schemeTableau10[((f[v]-3)/2) % 10]));
    const wastePct = w.reserved ? (100 * (w.reserved - w.used) / data.pool).toFixed(0) : 0;
    panels[key].label.text(`active ${w.active} · queued ${w.queued} · done ${w.done}`);
    panels[key].label2.text(`reserved-but-unused: ${w.reserved - w.used} blocks (${wastePct}% of pool)`);
  }
}
let t = 0, timer = null;
btn.on("click", () => {
  if (timer) { timer.stop(); timer = null; btn.text("▶ play"); }
  else { timer = d3.interval(() => { t = (t + 1) % T; render(t); }, 90); btn.text("⏸ pause"); }
});
scrub.on("input", function() { t = +this.value; render(t); });
render(0);
'''
show_d3(JS, alloc, height=270)

**What to notice, step by step.**
1. Scrub to ~step 40. The left pool is dominated by **red stripes**: memory promised to requests
   that haven't generated that far (most never will — average length is half of `max_len`). The
   right pool is nearly all solid color: paged blocks are allocated *when tokens actually arrive*.
2. Watch the **queued** counter. The contiguous allocator turns requests away while holding a pool
   full of stripes — that's memory-induced queueing, and it's why its `done` counter falls behind.
3. This is the entire vLLM thesis in one animation: **admission capacity = free blocks**, and paging
   keeps blocks free. Prefix caching (notebook 22) is these same blocks being *shared* between
   requests; block quantization (notebook 23's `--kv-cache-dtype fp8`) makes each square half price.

## Viz 4 · Speculative decoding, token by token

Notebook 24 gave you the formula `E[tokens/round] = (1-α^(k+1))/(1-α)`. Here you can *watch* the
rounds happen: the draft proposes k tokens (thin outline), the target verifies — green survive,
the first red kills the rest, blue is the free resample token every round gets.

In [ ]:
# All simulation happens in JS here (it's just coin flips) - Python only sets the defaults.
JS = r'''
const M = {left: 12, top: 8};
const controls = root.append("div").style("font-size", "13px");
function slider(label, min, max, value, step, fmt) {
  const wrap = controls.append("label").style("margin-right", "22px");
  wrap.append("span").text(label + " ");
  const out = wrap.append("b");
  const inp = wrap.append("input").attr("type", "range")
      .attr("min", min).attr("max", max).attr("step", step).attr("value", value)
      .style("vertical-align", "middle").style("margin-left", "6px");
  inp.on("input", () => { out.text(fmt(+inp.node().value)); reset(); });
  out.text(fmt(value));
  return inp;
}
const aS = slider("acceptance α:", 0.05, 0.95, +data.alpha, 0.05, v => v.toFixed(2));
const kS = slider("draft length k:", 1, 8, +data.k, 1, v => v);

const svg = root.append("svg").attr("width", W).attr("height", 210);
const stripG = svg.append("g").attr("transform", `translate(${M.left},70)`);
const stats = svg.append("text").attr("x", M.left).attr("y", 30).style("font-size", "13px");
const stats2 = svg.append("text").attr("x", M.left).attr("y", 50).style("font-size", "13px").style("fill", "#666");
const legend = svg.append("text").attr("x", M.left).attr("y", 200).style("font-size", "12px").style("fill", "#666")
    .text("green = draft token accepted · red = rejected (everything after dies) · blue = the guaranteed resample token");

const cell = 20; let tokens = [], rounds = 0, emitted = 0, timer = null;

function oneRound() {
  const alpha = +aS.node().value, k = +kS.node().value;
  let accepted = 0;
  while (accepted < k && Math.random() < alpha) accepted++;
  const rejected = accepted < k;
  for (let i = 0; i < accepted; i++) tokens.push("ok");
  if (rejected) tokens.push("rej");
  tokens.push("bonus");                       // resample (on reject) or target-sampled bonus token
  rounds++; emitted += accepted + 1;
  if (tokens.length > 3 * Math.floor((W - 30) / cell)) tokens = tokens.slice(-3 * Math.floor((W - 30) / cell));
  const perRow = Math.floor((W - 30) / cell);
  stripG.selectAll("rect").data(tokens).join("rect")
      .attr("x", (d, i) => (i % perRow) * cell)
      .attr("y", (d, i) => Math.floor(i / perRow) * (cell + 6))
      .attr("width", cell - 3).attr("height", cell - 3).attr("rx", 3)
      .attr("fill", d => d === "ok" ? "#66bb6a" : d === "rej" ? "#ef5350" : "#42a5f5");
  const alphaK = (1 - Math.pow(alpha, k + 1)) / (1 - alpha);
  stats.text(`rounds (target forward passes): ${rounds} · tokens emitted: ${emitted} · ` +
             `measured tokens/pass: ${(emitted / rounds).toFixed(2)}`);
  stats2.text(`theory says E[tokens/pass] = (1-α^(k+1))/(1-α) = ${alphaK.toFixed(2)} - watch the measured value converge`);
}
function reset() { tokens = []; rounds = 0; emitted = 0; stripG.selectAll("rect").remove(); }

const btn = controls.append("button").text("▶ run").style("margin-left", "8px");
btn.on("click", () => {
  if (timer) { timer.stop(); timer = null; btn.text("▶ run"); }
  else { timer = d3.interval(oneRound, 220); btn.text("⏸ pause"); }
});
for (let i = 0; i < 12; i++) oneRound();   // seed a few rounds so the cell is never blank on load
'''
show_d3(JS, {"alpha": 0.7, "k": 4}, height=230)

**What to notice, step by step.**
1. Run with α=0.7, k=4 and let the measured tokens/pass converge — it lands on the formula's value
   ((1−0.7⁵)/(1−0.7) ≈ **2.77**). The law of large numbers, live.
2. Drag α to 0.2: mostly red. Every red square after position 0 is *wasted draft compute* — this is
   the regime where speculation makes serving **slower** (notebook 24's right-hand plot, below the
   grey line).
3. Drag α to 0.9, k to 8: long green trains. This is what code completion feels like to a serving
   engine — and why speculative decoding was invented for it.

## Recap

| Viz | The one-sentence takeaway |
|---|---|
| KV explorer | Context length × architecture decides how many users fit on a GPU — before any code runs |
| Batching Gantt | Continuous batching wins exactly as much as your traffic's length-variance allows |
| Block pool | Paging converts "reserved but unused" memory into admitted requests |
| Speculation strip | α is destiny: measure it before you deploy a draft model |

The `show_d3` pattern (Python truth → JSON → D3 pixels) is deliberately reusable — the
[next notebook](./Serving_Logs_Observability.ipynb) uses it to turn **real vLLM logs and Prometheus
metrics** into the same kind of pictures, which is how you'd actually watch these mechanisms in
production.

### Further reading
- [D3 documentation](https://d3js.org/) · [Observable's D3 gallery](https://observablehq.com/@d3/gallery) — steal ideas from here
- The measured versions of everything above: notebooks [21](./Serving_Fundamentals_KV_Cache_Batching.ipynb) ·
  [22](./vLLM_High_Throughput_Serving.ipynb) · [23](./Quantized_Serving_Showdown.ipynb) ·
  [24](./Speculative_Decoding_Advanced_Serving.ipynb)

▶ **Next:** [Reading the Logs: vLLM Observability](./Serving_Logs_Observability.ipynb)